# ThreatLens AI - treinamento gratuito em GPU

Este notebook treina a baseline `hybrid_v2`, avalia o melhor peso no conjunto de teste e empacota as evidencias do experimento. Antes de executar, selecione **Runtime > Change runtime type > T4 GPU**.

In [ ]:
%pip install -q ultralytics==8.4.60 PyYAML>=6.0


## 1. Enviar e extrair o bundle

Escolha `artifacts/threatlens-training-bundle.zip` quando o seletor abrir.

In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
bundle_name = next(name for name in uploaded if name.endswith('.zip'))
with zipfile.ZipFile(bundle_name) as archive:
    archive.extractall('/content')
WORKSPACE = Path('/content/threatlens_training')
assert WORKSPACE.exists(), 'Bundle invalido: pasta threatlens_training ausente'
print(f'Workspace: {WORKSPACE}')


## 2. Validar GPU e tornar o YAML portatil

In [ ]:
import torch
import yaml

assert torch.cuda.is_available(), 'GPU nao encontrada. Ative o runtime T4 e reconecte.'
print(torch.cuda.get_device_name(0))
dataset_root = WORKSPACE / 'dataset' / 'hybrid_v2'
source_yaml = dataset_root / 'architecture.yaml'
portable_yaml = dataset_root / 'architecture-colab.yaml'
config = yaml.safe_load(source_yaml.read_text(encoding='utf-8'))
config['path'] = str(dataset_root.resolve())
portable_yaml.write_text(yaml.safe_dump(config, sort_keys=False), encoding='utf-8')
print(portable_yaml.read_text(encoding='utf-8'))


## 3. Configurar e treinar

`yolov8s.pt` e uma baseline mais forte que a versao nano e ainda cabe confortavelmente em uma T4. Os checkpoints sao salvos a cada 10 epocas.

In [ ]:
MODEL = 'yolov8s.pt'
EPOCHS = 100
IMAGE_SIZE = 640
BATCH = 8
RUN_NAME = 'threatlens-hybrid-v2-yolov8s'
RUNS_ROOT = WORKSPACE / 'training_runs'
EVALUATION_ROOT = WORKSPACE / 'evaluation_results'


In [ ]:
import subprocess
import sys

command = [
    sys.executable, str(WORKSPACE / 'scripts' / 'train_yolo.py'),
    '--data', str(portable_yaml),
    '--model', MODEL, '--device', '0',
    '--epochs', str(EPOCHS), '--imgsz', str(IMAGE_SIZE),
    '--batch', str(BATCH), '--workers', '2',
    '--patience', '25', '--save-period', '10',
    '--augmentation-profile', 'diagram', '--cache', 'disk', '--seed', '42',
    '--project', str(RUNS_ROOT), '--run-name', RUN_NAME,
]
subprocess.run(command, cwd=WORKSPACE, check=True)
RUN_DIR = RUNS_ROOT / RUN_NAME
BEST_MODEL = RUN_DIR / 'weights' / 'best.pt'
assert BEST_MODEL.exists(), f'Peso nao encontrado: {BEST_MODEL}'
print(f'Melhor peso: {BEST_MODEL}')


## 4. Avaliar no teste reservado

In [ ]:
evaluation_command = [
    sys.executable, str(WORKSPACE / 'scripts' / 'evaluate_model.py'),
    '--model', str(BEST_MODEL), '--data', str(portable_yaml),
    '--output-dir', str(EVALUATION_ROOT), '--split', 'test',
    '--device', '0', '--imgsz', str(IMAGE_SIZE), '--batch', str(BATCH),
]
subprocess.run(evaluation_command, cwd=WORKSPACE, check=True)

source_evaluation_command = [
    sys.executable, str(WORKSPACE / 'scripts' / 'evaluate_source_slices.py'),
    '--model', str(BEST_MODEL), '--data', str(portable_yaml),
    '--provenance', str(dataset_root / 'reports' / 'provenance.csv'),
    '--output-dir', str(EVALUATION_ROOT / 'source-slices'), '--split', 'test',
    '--device', '0', '--imgsz', str(IMAGE_SIZE), '--batch', str(BATCH),
]
subprocess.run(source_evaluation_command, cwd=WORKSPACE, check=True)


## 5. Baixar pesos, curvas e metricas

In [ ]:
import shutil

export_root = WORKSPACE / 'export'
if export_root.exists():
    shutil.rmtree(export_root)
shutil.copytree(RUN_DIR, export_root / 'training')
shutil.copytree(EVALUATION_ROOT, export_root / 'evaluation')

archive_base = Path('/content/threatlens-hybrid-v2-results')
archive_path = shutil.make_archive(str(archive_base), 'zip', root_dir=export_root)
print(f'Artefatos: {archive_path}')
files.download(archive_path)
